# Supervised Learning: House Value Regression

Predicts median house value using Linear Regression on the California Housing dataset.
Model performance is validated using R-squared (and supporting MAE/RMSE metrics).


In [ ]:
# --- Imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

sns.set_style("whitegrid")
RANDOM_STATE = 42

## 1. Config (no hardcoded paths/columns)

In [ ]:
# --- Config ---
DATA_PATH = "california_housing.csv"   # relative to this notebook
TARGET_COL = "MedHouseVal"
TEST_SIZE = 0.2

## 2. Load Data

In [ ]:
def load_data(path: str) -> pd.DataFrame:
    """Load California Housing data, caching to a local CSV so the
    notebook doesn't re-download on every run."""
    import os
    if os.path.exists(path):
        return pd.read_csv(path)

    dataset = fetch_california_housing(as_frame=True)
    df = dataset.frame.rename(columns={"MedHouseVal": TARGET_COL})
    df.to_csv(path, index=False)
    return df

df = load_data(DATA_PATH)
df.head()

## 3. Validation Checks

In [ ]:
def validate_dataframe(df: pd.DataFrame, target_col: str) -> None:
    """Sanity-check the dataset before modeling."""
    assert target_col in df.columns, f"Missing target column: {target_col}"
    assert df.isnull().sum().sum() == 0, "Unexpected nulls found in dataset"
    assert (df.select_dtypes(include=[np.number]).dtypes != object).all(), \
        "All numeric columns should have numeric dtype"
    assert (df[target_col] > 0).all(), "Target values should be positive (house values)"
    print("Validation passed: no nulls, correct dtypes, valid value ranges.")
    print(df.describe().T[["min", "max", "mean"]])

validate_dataframe(df, TARGET_COL)

## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[TARGET_COL], bins=40, ax=axes[0])
axes[0].set_title("Distribution of Median House Value")

corr = df.corr(numeric_only=True)[TARGET_COL].sort_values(ascending=False)
sns.barplot(x=corr.values, y=corr.index, ax=axes[1])
axes[1].set_title("Correlation with Target")
plt.tight_layout()
plt.show()

## 5. Train / Test Split + Scaling

In [ ]:
def split_and_scale(df: pd.DataFrame, target_col: str, test_size: float, random_state: int):
    """Split features/target and scale features. Returns train/test arrays + fitted scaler."""
    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, y_train, y_test, scaler, X.columns

X_train, X_test, y_train, y_test, scaler, feature_names = split_and_scale(
    df, TARGET_COL, TEST_SIZE, RANDOM_STATE
)
X_train.shape, X_test.shape

## 6. Train Model

In [ ]:
def train_linear_regression(X_train, y_train) -> LinearRegression:
    model = LinearRegression()
    model.fit(X_train, y_train)
    return model

model = train_linear_regression(X_train, y_train)
pd.Series(model.coef_, index=feature_names).sort_values(key=abs, ascending=False)

## 7. Evaluate (R-squared + supporting metrics)

In [ ]:
def evaluate_model(model, X_test, y_test) -> dict:
    """Return R2, MAE, RMSE for the given model on test data."""
    y_pred = model.predict(X_test)
    return {
        "r2": r2_score(y_test, y_pred),
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "y_pred": y_pred,
    }

metrics = evaluate_model(model, X_test, y_test)
print(f"R-squared: {metrics['r2']:.4f}")
print(f"MAE:       {metrics['mae']:.4f}")
print(f"RMSE:      {metrics['rmse']:.4f}")

assert 0 <= metrics["r2"] <= 1 or metrics["r2"] < 0, "R2 should be a valid float"  # sanity check, not a quality gate

## 8. Predicted vs Actual

In [ ]:
y_pred = metrics["y_pred"]

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.3, s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2)
plt.xlabel("Actual House Value")
plt.ylabel("Predicted House Value")
plt.title(f"Predicted vs Actual (R² = {metrics['r2']:.3f})")
plt.tight_layout()
plt.show()

## 9. Summary

- Model: Linear Regression on California Housing data (8 features, scaled).
- Note: Logistic Regression was not applied here since house value prediction is a
  continuous target (regression problem), not a classification problem — logistic
  regression applies to categorical/binary targets.
- R² reported above quantifies the proportion of variance in house value explained
  by the model.
